# Pre-entrega 6 · Demo del flujo de delegación

Este notebook muestra el orquestador multi-agente resolviendo una consulta que necesita **dos especialistas** y una **síntesis final**:

1. el **Supervisor** decide a quién delegar;
2. el **Investigador** busca las reglas en el manual del ERP (Pinecone, pre-entrega 4);
3. el **Analista** hace los cálculos con herramientas;
4. el Supervisor valida con su rúbrica y, si todo está completo, el **Sintetizador** redacta la respuesta.

> Requiere el `.env` con las keys de Anthropic, OpenAI (embeddings) y Pinecone. Ver el README.

In [ ]:
import json
from IPython.display import Image, Markdown, display

import config
from graph import build_graph
from runner import PREGUNTA_DEMO, ejecutar, guardar

print(f"LLM: {config.LLM_PROVIDER} · base de conocimiento: {config.RESEARCH_BACKEND} · "
      f"tope de delegaciones: {config.MAX_DELEGACIONES} · revisiones por especialista: {config.MAX_REVISIONES}")

## 1. El grafo

Topología jerárquica: todos los especialistas reportan al supervisor, y solo el supervisor decide el siguiente paso (líneas punteadas = aristas condicionales).

In [ ]:
graph = build_graph()
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:  # sin acceso a mermaid.ink: se muestra el código Mermaid
    print(graph.get_graph().draw_mermaid())

## 2. La consulta

Necesita **investigar** (validez de un presupuesto, seña de muebles a medida, quién aprueba descuentos) y después **calcular** (días transcurridos, precio con descuento, monto de la seña).

In [ ]:
print(PREGUNTA_DEMO)

## 3. Ejecución paso a paso

Cada bloque es un nodo del grafo: las decisiones del supervisor y lo que aporta cada especialista.

In [ ]:
traza = await ejecutar(graph, PREGUNTA_DEMO)

## 4. Quién aportó qué

El estado compartido registra cada contribución con su agente, ronda, encargo, herramientas y fuentes.

In [ ]:
print("Recorrido:", " → ".join(traza["recorrido"]), "→ END\n")

filas = ["| Ronda | Decisión del supervisor | Origen | Motivo |", "|---|---|---|---|"]
filas += [f"| {d['ronda']} | {d['siguiente']} | {d['origen']} | {d['motivo']} |" for d in traza["decisiones"]]
display(Markdown("\n".join(filas)))

filas = ["| Ronda | Agente | Herramientas | Fuentes |", "|---|---|---|---|"]
filas += [f"| {c['ronda']} | {c['agente']} | {', '.join(c['herramientas']) or '—'} | {', '.join(c['fuentes']) or '—'} |"
          for c in traza["contribuciones"]]
display(Markdown("\n".join(filas)))

## 5. Respuesta final

In [ ]:
display(Markdown(traza["respuesta_final"]))
guardar(traza)
print(f"\nDuración: {traza['duracion_segundos']} s · traza guardada en trazas/traza_ejemplo.json y .log")